In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="Versões dos pacotes">

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou mais recentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Você pode usar opções para personalizar a primitiva Estimator. Embora a interface do método `run()` das primitivas seja comum a todas as implementações, suas opções não são. Consulte as referências de API para obter informações sobre as opções de [`qiskit.primitives.BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) e [`qiskit_aer.BaseEstimatorV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html).

<span id="pass-options"></span>

## Definir opções do Estimator

Você pode definir opções ao inicializar o Estimator, após inicializar o Estimator, ou pode atualizar as opções depois que o Estimator foi inicializado. Para instruções de uso dessas técnicas, consulte o tópico [Introdução às opções](/guides/runtime-options-overview#options-precedence).

Além disso, você pode definir o valor `precision` no método `run()`, conforme descrito na seção a seguir.
<span id="run-method"></span>

### Método Run()

Os únicos valores que você pode passar a `run()` são os definidos na interface. Ou seja, `precision` para Estimator. Isso sobrescreve qualquer valor definido para `default_precision` na execução atual.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Estimator.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Caso especial: precision
O método `EstimatorV2.run` aceita dois argumentos: uma lista de PUBs, cada um dos quais pode especificar um valor de precision específico para o PUB, e um argumento keyword precision. Esses valores de precision fazem parte da interface de execução do Estimator e são independentes das opções do Runtime Estimator. Eles têm precedência sobre qualquer valor especificado como opção, a fim de cumprir com a abstração do Estimator.

No entanto, se `precision` não for especificado por nenhum PUB nem no argumento keyword run (ou se todos forem `None`), o valor de precision das opções é usado, especialmente `default_precision`.

Note que as opções do Estimator contêm tanto `default_shots` quanto `default_precision`. No entanto, como gate-twirling está habilitado por padrão, o produto de `num_randomizations` e `shots_per_randomization` tem precedência sobre essas duas opções.

Especificamente, para qualquer PUB do Estimator:

1. Se o PUB especifica precision, use esse valor.
2. Se o argumento keyword precision for especificado em `run`, use esse valor.
3. Se `twirling` estiver habilitado (True por padrão), o produto de `num_randomizations` e `shots_per_randomization`, conforme especificado nas [opções de twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options), é usado.
4. Se `estimator.options.default_shots` for especificado, use esse valor para controlar a quantidade de dados.
5. Se `estimator.options.default_precision` for especificado, use esse valor.

Por exemplo, se precision for especificado em todos os quatro lugares, o de maior precedência (precision especificado no PUB) é usado.

> **Note:** A precision escala inversamente com o uso. Ou seja, quanto menor a precision, mais tempo de QPU é necessário para executar.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after initialization
# This uses auto-complete.
estimator.options.default_precision = 0.01
# This does bulk update.
estimator.options.update(
    default_precision=0.02, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `precision` for Estimator. This overwrites any value set for `default_precision` for the current run.

In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

observable = SparsePauliOp("Z" * 3)

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)
isa_observable1 = observable.apply_layout(transpiled1.layout)
isa_observable2 = observable.apply_layout(transpiled2.layout)

estimator = Estimator(mode=backend)
# Default precision to use if not specified in run()
estimator.options.default_precision = 0.01
# Run two circuits, requiring a precision of .02 for both.
estimator.run(
    [(transpiled1, isa_observable1), (transpiled2, isa_observable2)],
    precision=0.02,
)

<RuntimeJobV2('d7amh42k86tc73a1sa20', 'estimator')>

<span id="no-error-mitigation"></span>
## Desativar toda mitigação e supressão de erros
Você pode desativar toda mitigação e supressão de erros se, por exemplo, estiver fazendo pesquisa sobre suas próprias técnicas de mitigação. Para fazer isso, defina `resilience_level = 0`.

Exemplo:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

observable = SparsePauliOp("Z" * 3)

circuit = random_iqp(3)
circuit.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

isa_circuit = pass_manager.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Setting precision during primitive initialization
estimator = Estimator(mode=backend, options={"default_precision": 0.05})

# Run with precision=0.02, overwriting the default.
estimator.run(
    [(isa_circuit, isa_observable1)],
    precision=0.02,
)

<RuntimeJobV2('d8286b00bvlc73d1vn50', 'estimator')>

<span id="options-table"></span>
## Opções disponíveis
A tabela a seguir documenta as opções da versão mais recente de `qiskit-ibm-runtime`. Para ver versões de opções mais antigas, visite a [referência de API do `qiskit-ibm-runtime`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime) e selecione uma versão anterior.

<Accordion>
<AccordionItem title="`default_shots`">

O número total de shots a usar por circuito por configuração.

**Opções**: Inteiro >= 0

**Padrão**: None

[Documentação de API `default_shots`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>

<AccordionItem title="`default_precision`">

A precision padrão a usar para qualquer PUB ou chamada `run()` que não especifique uma.

**Opções**: Float > 0

**Padrão**: 0.015625 (1 / sqrt(4096))

[Documentação de API `default_precision`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling`">

Controla as configurações de mitigação de erros de dynamical decoupling.

[Documentação de API `dynamical_decoupling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Opções**: `True`, `False`

**Padrão**: `False`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Opções**: `middle`, `edges`

**Padrão**: `middle`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Opções: `asap`, `alap`
Padrão: `alap`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.sequence_type`">

Opções: `XX`, `XpXm`, `XY4`
Padrão: `XX`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Opções: `True`, `False`
Padrão: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`environment`">

[Documentação de API `environment`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Função callable que recebe o `ID do job` e o `resultado do job`.

**Opções**: None

**Padrão**: None
  </AccordionItem>

<AccordionItem title="`environment.job_tags`">

Lista de tags.

**Opções**: None

**Padrão**: None
  </AccordionItem>

<AccordionItem title="`environment.log_level`">

**Opções**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Padrão**: WARNING
  </AccordionItem>

<AccordionItem title="`environment.private`">

**Opções**: `True`, `False`

**Padrão**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`execution`">

[Documentação de API `execution`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Se deve redefinir os qubits para o estado fundamental a cada shot.

**Opções**: `True`, `False`

**Padrão**: `True`
  </AccordionItem>

<AccordionItem title="`execution.rep_delay`">
O atraso entre uma medição e o circuito quântico subsequente.

**Opções**: Valor no intervalo fornecido por `backend.rep_delay_range`

**Padrão**: Dado por `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`max_execution_time`">
Limita por quanto tempo um job pode ser executado, em segundos. Consulte o guia de [tempo máximo de execução](/guides/max-execution-time) para detalhes.

**Opções**: Número inteiro de segundos no intervalo [1, 10800]

**Padrão**: 10800 (3 horas)
  </AccordionItem>

<AccordionItem title="`resilience`">
Opções avançadas de resiliência para ajustar a estratégia de resiliência.

[Documentação de API `resilience`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Opções para aprendizado de ruído por camada.

[Documentação de API `resilience.layer_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Opções**: list[int] de 2-10 valores no intervalo [0, 200]

**Padrão**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Opções**: None, Inteiro >= 1

**Padrão**: `4`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Opções**: Inteiro >= 1

**Padrão**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Opções**: Inteiro >= 1

**Padrão**: `128`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_model`">

**Opções**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Padrão**: None

  </AccordionItem>

<AccordionItem title="`resilience.measure_mitigation`">

**Opções**: `True`, `False`

**Padrão**: `True`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning`">

Opções para aprendizado de ruído de medição.

[Documentação de API `resilience.measure_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Opções**: Inteiro >= 1

**Padrão**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Opções**: Inteiro, `auto`

**Padrão**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.pec_mitigation`">

**Opções**: `True`, `False`

**Padrão**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.pec`">

Opções de mitigação de cancelamento probabilístico de erros.

[Documentação de API `resilience.pec`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>

<AccordionItem title="`resilience.pec.max_overhead`">

**Opções**: `None`, Inteiro >= 1

**Padrão**: `100`

  </AccordionItem>

<AccordionItem title="`resilience.pec.noise_gain`">

**Opções**: `auto`, float no intervalo [0, 1]

**Padrão**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.zne_mitigation`">

**Opções**: `True`, `False`

**Padrão**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.zne`">

[Documentação de API `resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>

<AccordionItem title="`resilience.zne.amplifier`">

**Opções**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Padrão**: `gate_folding`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Opções**: Lista de floats

**Padrão**: `[0, *noise_factors]`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolator`">

**Opções**: Um ou mais de: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Padrão**: `(exponential, linear)`

  </AccordionItem>

<AccordionItem title="`resilience.zne.noise_factors`">

**Opções**: Lista de floats; cada float >= 1

**Padrão**: `(1, 1.5, 2)` para `PEA`, e `(1, 3, 5)` caso contrário

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`resilience_level`">

Quanta resiliência construir contra erros. Níveis mais altos geram resultados mais precisos ao custo de tempos de processamento mais longos. Consulte a seção [níveis de resiliência](/guides/estimator-noise-management#resilience) no tópico Gerenciamento de ruído para saber mais.

**Opções**: `0`, `1`, `2`

**Padrão**: `1`

[Documentação de API `resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>

<AccordionItem title="`seed_estimator`">

**Opções**: Inteiro

**Padrão**: None

[`seed_estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>

<AccordionItem title="`simulator`">

Opções a passar ao simular um backend

[Documentação de API `simulator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Opções**: Lista de nomes de gates base para decompor

**Padrão**: O conjunto de todos os gates base suportados pelo [simulador Qiskit Aer](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>

<AccordionItem title="`simulator.coupling_map`">

**Opções**: Lista de interações bidirecionais de dois qubits

**Padrão**: None, o que implica nenhuma restrição de conectividade (conectividade total).

  </AccordionItem>

<AccordionItem title="`simulator.noise_model`">

**Opções**: [Qiskit Aer NoiseModel](/guides/build-noise-models), ou sua representação

**Padrão**: None

  </AccordionItem>

<AccordionItem title="`simulator.seed_simulator`">

**Opções**: Inteiro

**Padrão**: None

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`twirling`">

Opções de twirling

[Documentação de API `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Opções**: True, False

**Padrão**: False

  </AccordionItem>

<AccordionItem title="`twirling.enable_measure`">

**Opções**: True, False

**Padrão**: True

  </AccordionItem>

<AccordionItem title="`twirling.num_randomizations`">

**Opções**: `auto`, Inteiro >= 1

**Padrão**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.shots_per_randomization`">

**Opções**: `auto`, Inteiro >= 1

**Padrão**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.strategy`">

**Opções**: `active`, `active-circuit`, `active-accum`, `all`

**Padrão**: `active-accum`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`experimental`">

Opções experimentais, quando disponíveis.

  </AccordionItem>

</Accordion>
<span id="options-compatibility-table"></span>
## Compatibilidade de funcionalidades
Certas funcionalidades de runtime não podem ser usadas juntas em um único job. Clique na aba apropriada para ver a lista de funcionalidades incompatíveis com a funcionalidade selecionada:

<Tabs>

  <TabItem value="Fractional" label="Fractional gates">
  Incompatível com:
  - Gate twirling
  - PEA
  - PEC

  </TabItem>

  <TabItem value="ZNE" label="Gate-folding ZNE">
    Incompatível com:
  - PEA
  - PEC

  Pode não funcionar ao usar gates personalizados.
  </TabItem>
  <TabItem value="Twirling" label="Gate twirling">
  Incompatível com fractional gates ou com stretches.

  Outras observações:
  - O measurement twirling só pode ser aplicado a medições terminais.
  - Não funciona com entangladores não-Clifford.

  </TabItem>

  <TabItem value="PEA" label="PEA">
    Incompatível com:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </TabItem>

  <TabItem value="PEC" label="PEC">
    Incompatível com:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </TabItem>

</Tabs>
## Próximos passos
> **Tip:** - Revise o guia [Introdução às opções](/guides/runtime-options-overview).
> - Encontre mais detalhes sobre os métodos `EstimatorV2` na [referência de API do Estimator](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2).
> - Decida em qual [modo de execução](/guides/execution-modes) executar seu job.
> - Aprenda sobre [gerenciamento de ruído com Estimator](/guides/estimator-noise-management).

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access an IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.

<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>


<AccordionItem title="`default_precision`">

The default precision to use for any PUB or `run()` call that does not specify one.

**Choices**: Float > 0

**Default**: 0.015625 (1 / sqrt(4096))

[`default_precision` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">

[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Callable function that receives the `Job ID` and `Job result`.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">

[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800]

**Default**: 10800 (3 hours)
  </AccordionItem>


<AccordionItem title="`resilience`">
Advanced resilience options to fine tune the resilience strategy.

[`resilience` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options for learning layer noise.

[`resilience.layer_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choices**: list[int] of 2-10 values in the range [0, 200]

**Default**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choices**: None, Integer >= 1

**Default**: `4`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choices**: Integer >= 1

**Default**: `128`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_model`">

**Choices**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Default**: None

  </AccordionItem>



<AccordionItem title="`resilience.measure_mitigation`">

**Choices**: `True`, `False`

**Default**: `True`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning`">

Options for measurement noise learning.

[`resilience.measure_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choices**: Integer, `auto`

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.pec_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.pec`">

Probabilistic error cancellation mitigation options.

[`resilience.pec` API documentation](/docs/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>


<AccordionItem title="`resilience.pec.max_overhead`">

**Choices**: `None`, Integer >= 1


**Default**: `100`

  </AccordionItem>



<AccordionItem title="`resilience.pec.noise_gain`">

**Choices**: `auto`, float in the range [0, 1]

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.zne_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.zne`">

[`resilience.zne` API documentation](/docs/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>


<AccordionItem title="`resilience.zne.amplifier`">

**Choices**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Default**: `gate_folding`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choices**: List of floats

**Default**: `[0, *noise_factors]`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolator`">

**Choices**: One or more of: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Default**: `(exponential, linear)`

  </AccordionItem>



<AccordionItem title="`resilience.zne.noise_factors`">

**Choices**: List of floats; each float >= 1

**Default**: `(1, 1.5, 2)` for `PEA`, and `(1, 3, 5)` otherwise

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`resilience_level`">

How much resilience to build against errors. Higher levels generate more accurate results at the expense of longer processing times. See the [resilience levels](/docs/guides/estimator-noise-management#resilience) section in the Noise management topic to learn more.

**Choices**: `0`, `1`, `2`

**Default**: `1`

[`resilience_level` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>


<AccordionItem title="`seed_estimator`">

**Choices**: Integer

**Default**: None

[`seed_estimator`](/docs/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: True

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Accordion>
  <AccordionItem title="Fractional gates">
    Incompatible with:
  - Gate twirling
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate-folding ZNE">
    Might not work when using custom gates. Incompatible with:
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate twirling">
    Incompatible with:
  - Fractional gates
  - Stretches

  Other notes:
  - Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.
  </AccordionItem>
  <AccordionItem title="PEA">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </AccordionItem>
  <AccordionItem title="PEC">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </AccordionItem>
</Accordion>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `EstimatorV2` methods in the [Estimator API reference](/docs/api/qiskit-ibm-runtime/estimator-v2).
    - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
    - Learn about [noise management with Estimator](/docs/guides/estimator-noise-management).
</Admonition>